# Comparação metodológica: SetFit, TF-IDF, SVM e transformer tradicional

Este notebook compara os métodos **nos mesmos emails e nos mesmos folds externos**. O objetivo não é escolher o maior número desta execução: com poucas dezenas de exemplos, diferenças de uma ou duas previsões são ruído plausível. A decisão de produção deve ser confirmada num lote futuro, anotado sem voltar a ajustar os modelos.

Métodos incluídos:

- baseline de classe mais frequente;
- TF-IDF de palavras + regressão logística;
- TF-IDF de palavras + SVM linear;
- TF-IDF de caracteres + SVM linear (útil para idiomas, gralhas, referências e formatos de email);
- TF-IDF de palavras e caracteres combinados + SVM linear;
- SetFit atual, carregado apenas se o fingerprint, as labels e os folds coincidirem;
- fine-tuning tradicional do BERTimbau, sem SetFit (opcional porque é demorado).

**Nota conceptual:** TF-IDF é uma representação do texto; SVM e regressão logística são classificadores que podem usar essa representação. Na variante combinada, as matrizes de palavras e caracteres são concatenadas e ajustadas dentro de cada fold.

A métrica principal é macro-F1. Accuracy é apresentada, mas pode esconder erros numa classe. Todas as transformações TF-IDF são ajustadas dentro de cada fold, evitando fuga de vocabulário do teste para o treino.

In [21]:
import gc
import json
import os
import re
import sys
import tempfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Executa o notebook na raiz do projeto ou dentro de research/.')
sys.path.insert(0, str(ROOT))

from src.email_data import LABELS
from src.finetuning import (
    DEFAULT_EMAILS_DIR, DEFAULT_EXCEL, EmailDataset, clear_device_cache,
    dataset_fingerprint, load_examples, make_model, stratified_folds,
)

SEED = 42
N_FOLDS = 5
RUN_TRANSFORMER = False  # mudar para True para treinar o BERTimbau sem SetFit
TRANSFORMER_MODEL = 'neuralmind/bert-base-portuguese-cased'
TRANSFORMER_EPOCHS = 5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_LENGTH = 512

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pd.set_option('display.max_colwidth', 120)

## 1. Dados e auditoria mínima

O treino de produção exclui respostas (`in_reply_to` preenchido); este notebook reutiliza exatamente o mesmo carregador. A auditoria procura desequilíbrio, textos repetidos e comprimento. Um duplicado em folds diferentes pode tornar a avaliação otimista.

In [12]:
examples = load_examples(DEFAULT_EXCEL, [DEFAULT_EMAILS_DIR])
fingerprint = dataset_fingerprint(examples)
data = pd.DataFrame(examples, columns=['uid', 'text', 'label'])
data['n_chars'] = data['text'].str.len()
data['n_words'] = data['text'].str.split().str.len()
data['normalized_text'] = data['text'].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()

duplicate_groups = (
    data.groupby('normalized_text', sort=False)
        .filter(lambda group: len(group) > 1)
        [['uid', 'label', 'normalized_text']]
        .sort_values('normalized_text')
)
audit = pd.DataFrame({
    'valor': [
        len(data), data['uid'].nunique(), data['normalized_text'].nunique(),
        int(data['uid'].duplicated().sum()), int(len(duplicate_groups)),
        int(data['n_words'].median()), int(data['n_words'].max()),
    ]
}, index=[
    'emails', 'UIDs únicos', 'textos normalizados únicos', 'UIDs repetidos',
    'emails pertencentes a grupos de texto repetido', 'palavras (mediana)', 'palavras (máximo)',
])
display(data['label'].value_counts().rename('n').to_frame())
display(audit)
if len(duplicate_groups):
    display(duplicate_groups.drop(columns='normalized_text'))

,n
label,
SPAM,16
Pedido de Informação,14
Pedido de Encomenda,11


,valor
emails,41
UIDs únicos,41
textos normalizados únicos,40
UIDs repetidos,0
emails pertencentes a grupos de texto repetido,2
palavras (mediana),174
palavras (máximo),1215


,uid,label
6,26054,Pedido de Informação
7,26057,Pedido de Informação


## 2. Protocolo de avaliação

Mantemos `StratifiedKFold` com seed 42 para comparação direta com o treino atual. Cada UID é previsto uma única vez fora do treino. Isto mede generalização para uma mistura aleatória dos emails observados; **não** simula perfeitamente os emails futuros.

Recomendação para a decisão final: congelar agora os candidatos e usar os próximos 30–50 emails anotados como teste prospetivo. Não ajustar hiperparâmetros nesse lote. Se começarem a existir várias mensagens do mesmo tópico/remetente, substituir a divisão por `StratifiedGroupKFold` e manter cada grupo num único fold.

In [13]:
folds = stratified_folds(examples, N_FOLDS, SEED)
fold_by_uid = {
    uid: fold_index
    for fold_index, fold in enumerate(folds, start=1)
    for uid, _text, _label in fold
}
assert len(fold_by_uid) == len(examples) == data['uid'].nunique()

fold_distribution = pd.crosstab(
    pd.Series(fold_by_uid, name='fold'),
    data.set_index('uid')['label'].rename('label'),
).reindex(columns=LABELS, fill_value=0)
display(fold_distribution)

label,Pedido de Informação,Pedido de Encomenda,SPAM
fold,,,
1,2,3,4
2,3,2,3
3,3,2,3
4,3,2,3
5,3,2,3


## 3. Baselines TF-IDF

Os hiperparâmetros abaixo ficam fixos antes de observar os folds de teste. Se forem afinados, a afinação deve acontecer numa validação interna de cada fold (*nested cross-validation*), nunca escolhendo o melhor resultado nesta tabela e apresentando-o como estimativa imparcial.

In [22]:
def evaluate_sklearn(display_name, estimator):
    rows = []
    for fold_index, test_examples in enumerate(folds, start=1):
        test_uids = {uid for uid, _text, _label in test_examples}
        train_examples = [row for row in examples if row[0] not in test_uids]
        model = clone(estimator)
        model.fit(
            [text for _uid, text, _label in train_examples],
            [label for _uid, _text, label in train_examples],
        )
        predicted = model.predict([text for _uid, text, _label in test_examples])
        rows.extend({
            'model': display_name, 'uid': uid, 'fold': fold_index,
            'label_real': label, 'label_prevista': str(prediction),
        } for (uid, _text, label), prediction in zip(test_examples, predicted))
    result = pd.DataFrame(rows).sort_values('uid').reset_index(drop=True)
    assert len(result) == len(examples) and result['uid'].nunique() == len(examples)
    return result

word_tfidf = dict(
    lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
    min_df=1, max_df=0.98, max_features=50_000, sublinear_tf=True,
)
char_tfidf = dict(
    analyzer='char_wb', lowercase=True, ngram_range=(3, 5),
    min_df=2, max_features=80_000, sublinear_tf=True,
)

classical_models = {
    'Dummy — classe mais frequente': DummyClassifier(strategy='most_frequent'),
    'TF-IDF palavras + regressão logística': Pipeline([
        ('tfidf', TfidfVectorizer(**word_tfidf)),
        ('classifier', LogisticRegression(C=1.0, class_weight='balanced', max_iter=3_000)),
    ]),
    'TF-IDF palavras + SVM linear': Pipeline([
        ('tfidf', TfidfVectorizer(**word_tfidf)),
        ('classifier', LinearSVC(C=1.0, class_weight='balanced')),
    ]),
    'TF-IDF caracteres + SVM linear': Pipeline([
        ('tfidf', TfidfVectorizer(**char_tfidf)),
        ('classifier', LinearSVC(C=1.0, class_weight='balanced')),
    ]),
    'TF-IDF palavras + caracteres + SVM linear': Pipeline([
        ('features', FeatureUnion([
            ('words', TfidfVectorizer(**word_tfidf)),
            ('characters', TfidfVectorizer(**char_tfidf)),
        ])),
        ('classifier', LinearSVC(C=1.0, class_weight='balanced')),
    ]),
}

prediction_frames = [
    evaluate_sklearn(name, model) for name, model in classical_models.items()
] 

## 4. Carregar o SetFit atual com validação forte

Não basta carregar o CSV mais recente: a base diária pode ter mudado. A célula seguinte recusa resultados se o fingerprint, o conjunto de UIDs, as labels reais ou a atribuição aos folds não coincidirem exatamente. Nesse caso, executar primeiro `python -m src.finetuning`.

In [23]:
def project_path(value):
    path = Path(value)
    candidate = path if path.is_absolute() else ROOT / path
    if candidate.exists():
        return candidate
    # latest.json pode ter sido criado noutra máquina com um caminho absoluto.
    if 'data' in path.parts:
        portable = ROOT.joinpath(*path.parts[path.parts.index('data'):])
        if portable.exists():
            return portable
    return candidate

def load_verified_setfit_predictions():
    pointer_path = ROOT / 'data/finetuning_results/latest.json'
    if not pointer_path.exists():
        raise FileNotFoundError('Não existe latest.json; executa python -m src.finetuning.')
    pointer = json.loads(pointer_path.read_text(encoding='utf-8'))
    metrics = json.loads(project_path(pointer['metrics']).read_text(encoding='utf-8'))
    if metrics.get('training_method') != 'setfit':
        raise ValueError('A execução mais recente não é SetFit.')
    if metrics.get('dataset_fingerprint') != fingerprint:
        raise ValueError('Os resultados SetFit são de outra versão dos dados.')
    source = pd.read_csv(project_path(pointer['run_dir']) / 'predictions.csv', dtype={'uid': str})
    source = source.rename(columns={'label_real': 'label_real', 'label_prevista': 'label_prevista'})
    expected_truth = dict(zip(data['uid'], data['label']))
    if set(source['uid']) != set(expected_truth) or source['uid'].nunique() != len(examples):
        raise ValueError('A cobertura de UIDs do SetFit não coincide com os dados atuais.')
    if dict(zip(source['uid'], source['label_real'])) != expected_truth:
        raise ValueError('As labels reais guardadas no SetFit não coincidem.')
    if dict(zip(source['uid'], source['fold'].astype(int))) != fold_by_uid:
        raise ValueError('Os folds guardados no SetFit não coincidem.')
    source['model'] = 'BERTimbau + SetFit (atual)'
    return source[['model', 'uid', 'fold', 'label_real', 'label_prevista']]

setfit_predictions = load_verified_setfit_predictions()
prediction_frames.append(setfit_predictions)
print(f'SetFit validado: {len(setfit_predictions)} previsões da execução mais recente.')

SetFit validado: 41 previsões da execução mais recente.


## 5. Fine-tuning tradicional do transformer (sem SetFit)

Muda `RUN_TRANSFORMER = True` na configuração e executa novamente. Cada fold começa no checkpoint BERTimbau original e aprende diretamente uma cabeça de três classes. O teste externo não é usado para *early stopping* nem para escolher épocas. Recomenda-se CUDA; em CPU serão cinco treinos demorados.

In [24]:
def transformer_cross_validation():
    import torch
    from transformers import AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

    if torch.cuda.is_available():
        device_name = 'cuda'
    elif torch.backends.mps.is_available():
        device_name = 'mps'
    else:
        device_name = 'cpu'
    print(f'Dispositivo: {device_name}')

    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    collator = DataCollatorWithPadding(tokenizer)
    rows = []
    for fold_index, test_examples in enumerate(folds, start=1):
        test_uids = {uid for uid, _text, _label in test_examples}
        train_examples = [row for row in examples if row[0] not in test_uids]
        fold_seed = SEED + fold_index - 1
        set_seed(fold_seed)
        train_dataset = EmailDataset(train_examples, tokenizer, MAX_LENGTH)
        test_dataset = EmailDataset(test_examples, tokenizer, MAX_LENGTH)

        with tempfile.TemporaryDirectory(prefix=f'transformer-fold-{fold_index}-') as temporary:
            model = make_model(TRANSFORMER_MODEL)
            use_bf16 = device_name == 'cuda' and torch.cuda.is_bf16_supported()
            args = TrainingArguments(
                output_dir=temporary, use_cpu=device_name == 'cpu',
                num_train_epochs=TRANSFORMER_EPOCHS,
                per_device_train_batch_size=TRAIN_BATCH_SIZE,
                per_device_eval_batch_size=EVAL_BATCH_SIZE,
                learning_rate=LEARNING_RATE, weight_decay=0.01,
                eval_strategy='no', save_strategy='no', logging_strategy='no',
                report_to='none', disable_tqdm=False,
                fp16=device_name == 'cuda' and not use_bf16, bf16=use_bf16,
                seed=fold_seed, data_seed=fold_seed,
            )
            trainer = Trainer(
                model=model, args=args, train_dataset=train_dataset,
                data_collator=collator, processing_class=tokenizer,
            )
            trainer.train()
            output = trainer.predict(test_dataset)
            predicted = np.argmax(output.predictions, axis=-1)
            rows.extend({
                'model': 'BERTimbau — fine-tuning tradicional',
                'uid': uid, 'fold': fold_index, 'label_real': label,
                'label_prevista': LABELS[int(prediction)],
            } for (uid, _text, label), prediction in zip(test_examples, predicted))
            del output, trainer, model, train_dataset, test_dataset
            clear_device_cache()
            gc.collect()

    result = pd.DataFrame(rows).sort_values('uid').reset_index(drop=True)
    assert len(result) == len(examples) and result['uid'].nunique() == len(examples)
    return result

if RUN_TRANSFORMER:
    prediction_frames.append(transformer_cross_validation())
else:
    print('Transformer não executado. Define RUN_TRANSFORMER=True para o treinar.')

Transformer não executado. Define RUN_TRANSFORMER=True para o treinar.


## 6. Resultados e incerteza

A tabela usa as previsões out-of-fold agregadas. Os intervalos por bootstrap são apenas descritivos: folds sobrepostos e uma amostra tão pequena violam a independência perfeita. A comparação decisiva continua a ser o lote futuro congelado.

In [25]:
predictions = pd.concat(prediction_frames, ignore_index=True)

def metric_row(model_name, group):
    truth = group['label_real']
    predicted = group['label_prevista']
    precision, recall, f1, support = precision_recall_fscore_support(
        truth, predicted, labels=LABELS, zero_division=0,
    )
    row = {
        'Modelo': model_name,
        'Accuracy': accuracy_score(truth, predicted),
        'Macro-F1': f1_score(truth, predicted, labels=LABELS, average='macro', zero_division=0),
    }
    for index, label in enumerate(LABELS):
        row[f'Recall — {label}'] = recall[index]
        row[f'F1 — {label}'] = f1[index]
    return row

summary = pd.DataFrame([metric_row(name, group) for name, group in predictions.groupby('model', sort=False)])
summary = summary.sort_values('Macro-F1', ascending=False).reset_index(drop=True)
display(summary.style.format({column: '{:.3f}' for column in summary.columns if column != 'Modelo'}))

,Modelo,Accuracy,Macro-F1,Recall — Pedido de Informação,F1 — Pedido de Informação,Recall — Pedido de Encomenda,F1 — Pedido de Encomenda,Recall — SPAM,F1 — SPAM
0,TF-IDF caracteres + SVM linear,0.585,0.579,0.571,0.571,0.545,0.522,0.625,0.645
1,TF-IDF palavras + caracteres + SVM linear,0.585,0.575,0.500,0.538,0.545,0.500,0.688,0.688
2,BERTimbau + SetFit (atual),0.585,0.573,0.714,0.645,0.455,0.455,0.562,0.621
3,TF-IDF palavras + SVM linear,0.512,0.501,0.429,0.462,0.455,0.435,0.625,0.606
4,TF-IDF palavras + regressão logística,0.415,0.415,0.357,0.345,0.455,0.435,0.438,0.467
5,Dummy — classe mais frequente,0.341,0.225,0.143,0.174,0.000,0.000,0.750,0.500


In [26]:
def paired_bootstrap_against_setfit(candidate_name, n_resamples=5_000):
    reference = predictions[predictions['model'] == 'BERTimbau + SetFit (atual)']
    candidate = predictions[predictions['model'] == candidate_name]
    paired = reference.merge(candidate, on='uid', suffixes=('_setfit', '_candidate'), validate='one_to_one')
    assert (paired['label_real_setfit'] == paired['label_real_candidate']).all()
    truth = paired['label_real_setfit'].to_numpy()
    setfit_pred = paired['label_prevista_setfit'].to_numpy()
    candidate_pred = paired['label_prevista_candidate'].to_numpy()
    rng = np.random.default_rng(SEED)
    deltas = np.empty((n_resamples, 2))
    for i in range(n_resamples):
        index = rng.integers(0, len(paired), len(paired))
        deltas[i, 0] = accuracy_score(truth[index], candidate_pred[index]) - accuracy_score(truth[index], setfit_pred[index])
        deltas[i, 1] = f1_score(truth[index], candidate_pred[index], labels=LABELS, average='macro', zero_division=0) - f1_score(truth[index], setfit_pred[index], labels=LABELS, average='macro', zero_division=0)
    return {
        'Modelo vs SetFit': candidate_name,
        'Delta accuracy': deltas[:, 0].mean(),
        'IC95% delta accuracy': tuple(np.quantile(deltas[:, 0], [0.025, 0.975]).round(3)),
        'Delta macro-F1': deltas[:, 1].mean(),
        'IC95% delta macro-F1': tuple(np.quantile(deltas[:, 1], [0.025, 0.975]).round(3)),
    }

comparison_names = [name for name in predictions['model'].unique() if name not in {
    'BERTimbau + SetFit (atual)', 'Dummy — classe mais frequente',
}]
uncertainty = pd.DataFrame([paired_bootstrap_against_setfit(name) for name in comparison_names])
display(uncertainty)

,Modelo vs SetFit,Delta accuracy,IC95% delta accuracy,Delta macro-F1,IC95% delta macro-F1
0,TF-IDF palavras + regressão logística,-0.172390,"(-0.341, 0.0)",-0.159607,"(-0.339, 0.017)"
1,TF-IDF palavras + SVM linear,-0.073073,"(-0.22, 0.073)",-0.073092,"(-0.217, 0.065)"
2,TF-IDF caracteres + SVM linear,-0.001405,"(-0.171, 0.171)",0.004336,"(-0.159, 0.173)"
3,TF-IDF palavras + caracteres + SVM linear,-0.001127,"(-0.146, 0.122)",0.000373,"(-0.129, 0.132)"


In [27]:
for model_name, group in predictions.groupby('model', sort=False):
    matrix = confusion_matrix(group['label_real'], group['label_prevista'], labels=LABELS)
    display(pd.DataFrame(matrix, index=[f'Real: {x}' for x in LABELS], columns=[f'Prev.: {x}' for x in LABELS]).style.set_caption(model_name))

error_table = (
    predictions.assign(correto=lambda frame: frame['label_real'] == frame['label_prevista'])
        .pivot(index=['uid', 'label_real'], columns='model', values='label_prevista')
        .reset_index()
)
display(error_table)

,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,2,0,12
Real: Pedido de Encomenda,3,0,8
Real: SPAM,4,0,12


,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,5,4,5
Real: Pedido de Encomenda,4,5,2
Real: SPAM,6,3,7


,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,6,4,4
Real: Pedido de Encomenda,3,5,3
Real: SPAM,3,3,10


,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,8,3,3
Real: Pedido de Encomenda,3,6,2
Real: SPAM,3,3,10


,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,7,4,3
Real: Pedido de Encomenda,3,6,2
Real: SPAM,2,3,11


,Prev.: Pedido de Informação,Prev.: Pedido de Encomenda,Prev.: SPAM
Real: Pedido de Informação,10,2,2
Real: Pedido de Encomenda,4,5,2
Real: SPAM,3,4,9


model,uid,label_real,BERTimbau + SetFit (atual),Dummy — classe mais frequente,TF-IDF caracteres + SVM linear,TF-IDF palavras + SVM linear,TF-IDF palavras + caracteres + SVM linear,TF-IDF palavras + regressão logística
0,22545,SPAM,SPAM,Pedido de Informação,SPAM,SPAM,SPAM,SPAM
1,22561,SPAM,SPAM,Pedido de Informação,SPAM,SPAM,SPAM,SPAM
2,25980,Pedido de Informação,Pedido de Encomenda,SPAM,Pedido de Informação,Pedido de Encomenda,Pedido de Encomenda,Pedido de Encomenda
3,26004,Pedido de Encomenda,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação
4,26007,Pedido de Encomenda,Pedido de Informação,SPAM,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação
5,26016,Pedido de Informação,Pedido de Informação,SPAM,Pedido de Encomenda,Pedido de Encomenda,Pedido de Encomenda,Pedido de Encomenda
6,26054,Pedido de Informação,Pedido de Informação,SPAM,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação
7,26057,Pedido de Informação,Pedido de Informação,SPAM,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação
8,26505,Pedido de Informação,SPAM,SPAM,SPAM,SPAM,SPAM,SPAM
9,26666,Pedido de Encomenda,Pedido de Informação,SPAM,Pedido de Informação,Pedido de Informação,Pedido de Informação,Pedido de Informação


## 7. Guardar a execução

São guardadas todas as previsões, o resumo e a configuração. Isto permite auditar diferenças por UID e impede que uma tabela isolada perca o contexto experimental.

In [28]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = ROOT / 'data/comparison_benchmark' / run_id
output_dir.mkdir(parents=True, exist_ok=False)
predictions.to_csv(output_dir / 'predictions.csv', index=False, encoding='utf-8-sig')
summary.to_csv(output_dir / 'summary.csv', index=False, encoding='utf-8-sig')
uncertainty.to_csv(output_dir / 'uncertainty_vs_setfit.csv', index=False, encoding='utf-8-sig')
(output_dir / 'configuration.json').write_text(json.dumps({
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_fingerprint': fingerprint, 'n_examples': len(examples),
    'labels': list(LABELS), 'seed': SEED, 'folds': N_FOLDS,
    'run_transformer': RUN_TRANSFORMER, 'transformer_model': TRANSFORMER_MODEL,
    'transformer_epochs': TRANSFORMER_EPOCHS, 'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH, 'folds_by_uid': fold_by_uid,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Resultados guardados em: {output_dir}')

Resultados guardados em: /home/alex/Desktop/classification_globalbrico/data/comparison_benchmark/20260928T095930Z


## Como interpretar

1. Exigir cobertura completa e o mesmo fingerprint/folds para todos os métodos.
2. Olhar primeiro para macro-F1 e recall por classe; usar accuracy apenas como complemento.
3. Tratar diferenças de uma ou duas previsões como inconclusivas. Rever esses UIDs e a consistência das regras de anotação.
4. Preferir o modelo mais simples se ficar estatisticamente empatado: TF-IDF + SVM treina em segundos, é barato e é fácil de inspecionar.
5. Só promover um candidato depois de vencer num lote temporal futuro, congelado.
6. Se a fronteira entre “Pedido de Informação” e “Pedido de Encomenda” continuar ambígua para humanos, escrever regras operacionais e medir concordância entre dois anotadores numa amostra. Mais exemplos inconsistentes não resolvem ambiguidade de label.